In [6]:
# -*- coding: utf-8 -*-

"""
AdaBoosting & XGBoost Benchmark
--------------------------------
Compares AdaBoost and XGBoost on the PlacementPredict dataset.

Dataset is stored in Google Drive.
"""


# ==========================================================================
# 1. Mount Google Drive
# ==========================================================================

from google.colab import drive

drive.mount('/content/drive')


# ==========================================================================
# 2. Import libraries
# ==========================================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier


# ==========================================================================
# 3. Configuration
# ==========================================================================

RANDOM_STATE = 42

TARGET_COL = "PlacementStatus"

# EXACT path of your dataset in Google Drive
DATA_PATH = "/content/drive/MyDrive/colab/placement_predict_50k Dataset (1).csv"


# ==========================================================================
# 4. Load dataset
# ==========================================================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# ==========================================================================
# 5. Drop IsAnomaly column
# ==========================================================================

# IsAnomaly is a data-quality marker and is not used as a feature.

if "IsAnomaly" in df.columns:

    df = df.drop(columns=["IsAnomaly"])

    print("IsAnomaly column dropped.")


# ==========================================================================
# 6. Separate target and features
# ==========================================================================

y = df[TARGET_COL].astype(int)

X = df.drop(columns=[TARGET_COL])


print("\nTarget column:", TARGET_COL)
print("Number of features:", X.shape[1])


# ==========================================================================
# 7. Identify categorical and numerical columns
# ==========================================================================

cat_cols = X.select_dtypes(
    exclude="number"
).columns.tolist()

num_cols = X.select_dtypes(
    include="number"
).columns.tolist()


print("\nCategorical columns:")
print(cat_cols)

print("\nNumerical columns:")
print(num_cols)


# ==========================================================================
# 8. Encode categorical columns
# ==========================================================================

encoders = {}

for c in cat_cols:

    le = LabelEncoder()

    X[c] = le.fit_transform(
        X[c].astype(str)
    )

    encoders[c] = le


# ==========================================================================
# 9. Handle missing numerical values
# ==========================================================================

imputer = SimpleImputer(
    strategy="median"
)

X[num_cols] = imputer.fit_transform(
    X[num_cols]
)


# ==========================================================================
# 10. Scale numerical features
# ==========================================================================

scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(
    X[num_cols]
)


# ==========================================================================
# 11. Train / Validation / Test Split
# ==========================================================================

VAL_SIZE = 0.15

TEST_SIZE = 0.15


# --------------------------------------------------------------------------
# First split:
# 85% = training + validation
# 15% = test
# --------------------------------------------------------------------------

X_train_val, X_test, y_train_val, y_test = train_test_split(

    X,
    y,

    test_size=TEST_SIZE,

    stratify=y,

    random_state=RANDOM_STATE
)


# --------------------------------------------------------------------------
# Second split:
# 70% = training
# 15% = validation
# --------------------------------------------------------------------------

val_ratio = VAL_SIZE / (1 - TEST_SIZE)


X_train, X_val, y_train, y_val = train_test_split(

    X_train_val,
    y_train_val,

    test_size=val_ratio,

    stratify=y_train_val,

    random_state=RANDOM_STATE
)


# ==========================================================================
# 12. Display dataset split
# ==========================================================================

print("\n" + "=" * 60)

print("DATA SPLIT")

print("=" * 60)

print("Training set   :", X_train.shape)

print("Validation set :", X_val.shape)

print("Test set       :", X_test.shape)


# ==========================================================================
# 13. Benchmark function
# ==========================================================================

def boosting_benchmark(
    X_train,
    y_train,
    X_val,
    y_val
):

    results = []


    # ======================================================================
    # MODEL 1: AdaBoost
    # ======================================================================

    print("\nTraining AdaBoost...")


    # Decision Tree acts as the weak learner.
    ada_base = DecisionTreeClassifier(

        max_depth=2,

        random_state=RANDOM_STATE
    )


    # AdaBoost repeatedly trains weak learners
    # and gives more importance to incorrectly
    # classified samples.

    ada = AdaBoostClassifier(

        estimator=ada_base,

        n_estimators=200,

        learning_rate=0.5,

        random_state=RANDOM_STATE
    )


    # Start timer
    t0 = time.time()


    # Train AdaBoost
    ada.fit(

        X_train,

        y_train
    )


    # Calculate training time
    ada_fit_time = time.time() - t0


    # Validation predictions
    ada_val_pred = ada.predict(
        X_val
    )


    # Validation probabilities
    ada_val_proba = ada.predict_proba(
        X_val
    )[:, 1]


    # Store AdaBoost results
    results.append({

        "model": "AdaBoost",

        "val_accuracy": accuracy_score(
            y_val,
            ada_val_pred
        ),

        "val_f1": f1_score(
            y_val,
            ada_val_pred
        ),

        "val_roc_auc": roc_auc_score(
            y_val,
            ada_val_proba
        ),

        "best_n_estimators": ada.n_estimators,

        "fit_time_sec": round(
            ada_fit_time,
            2
        )
    })


    # ======================================================================
    # MODEL 2: XGBoost
    # ======================================================================

    print("Training XGBoost...")


    xgb = XGBClassifier(

        n_estimators=1000,

        max_depth=4,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        early_stopping_rounds=30,

        random_state=RANDOM_STATE,

        n_jobs=-1
    )


    # Start timer
    t0 = time.time()


    # Train XGBoost
    xgb.fit(

        X_train,

        y_train,

        eval_set=[
            (X_val, y_val)
        ],

        verbose=False
    )


    # Calculate training time
    xgb_fit_time = time.time() - t0


    # Validation predictions
    xgb_val_pred = xgb.predict(
        X_val
    )


    # Validation probabilities
    xgb_val_proba = xgb.predict_proba(
        X_val
    )[:, 1]


    # Store XGBoost results
    results.append({

        "model": "XGBoost",

        "val_accuracy": accuracy_score(
            y_val,
            xgb_val_pred
        ),

        "val_f1": f1_score(
            y_val,
            xgb_val_pred
        ),

        "val_roc_auc": roc_auc_score(
            y_val,
            xgb_val_proba
        ),

        "best_n_estimators": xgb.best_iteration + 1,

        "fit_time_sec": round(
            xgb_fit_time,
            2
        )
    })


    # ======================================================================
    # Create leaderboard
    # ======================================================================

    results_df = pd.DataFrame(
        results
    )


    results_df = results_df.sort_values(

        "val_accuracy",

        ascending=False

    ).reset_index(
        drop=True
    )


    return results_df


# ==========================================================================
# 14. Run benchmark
# ==========================================================================

leaderboard = boosting_benchmark(

    X_train,

    y_train,

    X_val,

    y_val
)


# ==========================================================================
# 15. Display validation leaderboard
# ==========================================================================

print("\n" + "=" * 70)

print("VALIDATION LEADERBOARD")

print("=" * 70)


print(
    leaderboard.to_string(
        index=False
    )
)


# ==========================================================================
# 16. Save results to Google Drive
# ==========================================================================

OUTPUT_PATH = (
    "/content/drive/MyDrive/colab/"
    "boosting_benchmark_results.csv"
)


leaderboard.to_csv(

    OUTPUT_PATH,

    index=False
)


print("\n" + "=" * 70)

print("RESULTS SAVED")

print("=" * 70)


print(
    "Saved to:",
    OUTPUT_PATH
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!
Dataset shape: (50000, 32)
IsAnomaly column dropped.

Target column: PlacementStatus
Number of features: 30

Categorical columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA_Tier']

Numerical columns:
['StudentID', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'Salary Package']

DATA SPLIT
Training set   : (34999, 30)
Validation set : (7501, 30)
Test set       : (7500, 30)

Training AdaBoost...
Training XGBoost...

VALIDATION LEADERBOARD
   model  val_accuracy   val_f1  val_roc_auc  best_n_estimators  fit_time_sec
 XGBo